# Imports

In [1]:
import h5py
import numpy as np
import sys
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

# Data Loader

In [2]:
def to_categorical(labels, num_classes=None):
    """One-hot encodes class labels.

    **Arguments**

    - **labels** : _1-d numpy.ndarray_
        - Labels in the range `[0,num_classes)`.
    - **num_classes** : {_int_, `None`}
        - The total number of classes. If `None`, taken to be the 
        maximum label plus one.

    **Returns**

    - _2-d numpy.ndarray_
        - The one-hot encoded labels.
    """

    # get num_classes from max label if None
    if num_classes is None:
        num_classes = np.int(np.max(labels)) + 1

    y = np.asarray(labels, dtype=int)
    n = y.shape[0]
    categorical = np.zeros((n, num_classes))

    # index into array and set appropriate values to 1
    categorical[np.arange(n), y] = 1

    return categorical


def load_h5(h5_filename,mode='class',unsup=False, glob=False,nevts=-1):

    global_pl = []
    f = h5py.File(h5_filename,'r')
    nevts=int(nevts)
    data = f['data']

    if mode == 'class':
        label = f['pid']
        print("label", label)
        label_np = np.array(label)
        print("label_np", label_np)

        label_train = np.where(label_np > 1, 0, label_np)

        # label_train = label_np[label_np > 1] = 0
        print("label_train", label_train)

    elif mode == 'seg':
        label = f['label']
    else:
        print('No mode found')
    
    if glob:
        global_pl = f['global']
        return (data, label, global_pl)

    print("loaded {0} events".format(data.shape[1]))

    data = [data[key] for key in range(data.shape[0])]
    print("data", len(data))

    # label = to_categorical(label, num_classes=2)
    label = to_categorical(label_train, num_classes=2)
    print("label", label)

    return (data, label)  


def load_data():

    X_train, Y_train = load_h5('train_cnn1d_splitted.h5')
    X_val, Y_val = load_h5('validation_cnn1d_splitted.h5')
    X_test, Y_test = load_h5('test_cnn1d_splitted.h5')

    return X_train, X_val, X_test, Y_train, Y_val, Y_test

"""
Note that process types are tagged as follows:
    ttbarZ : 5
    ttbarWW : 4
    ttbarW : 3
    ttbarHiggs : 2
    4top : 1
"""

X_train, X_val, X_test, Y_train, Y_val, Y_test = load_data()



#print("X_train", X_train)
# print("X_val", X_val)
# print("X_test", X_test)

# print("Y_train", Y_train)
# print("Y_val", Y_val)
# print("Y_test", Y_test)


print("len of X_train", len(X_train))
print("len of X_val", len(X_val))
#print("len of X_test", len(X_test))

#print("len of Y_train", len(Y_train))
#print("len of Y_val", len(Y_val))
#print("len of Y_test", len(Y_test))

print("Type of X_train", type(X_train))
print("Type of X_val", type(X_val))

print("Shape of X_train", np.shape(X_train))
print("Shape of X_val", np.shape(X_val))

print("Shape of Y_train", np.shape(Y_train))
print("Shape of Y_val", np.shape(Y_val))


label <HDF5 dataset "pid": shape (241658,), type "<f8">
label_np [4. 2. 1. ... 1. 3. 3.]
label_train [0. 0. 1. ... 1. 0. 0.]
loaded 241658 events
data 11
label [[1. 0.]
 [1. 0.]
 [0. 1.]
 ...
 [0. 1.]
 [1. 0.]
 [1. 0.]]
label <HDF5 dataset "pid": shape (30207,), type "<f8">
label_np [5. 2. 1. ... 1. 2. 5.]
label_train [0. 0. 1. ... 1. 0. 0.]
loaded 30207 events
data 11
label [[1. 0.]
 [1. 0.]
 [0. 1.]
 ...
 [0. 1.]
 [1. 0.]
 [1. 0.]]
label <HDF5 dataset "pid": shape (30207,), type "<f8">
label_np [4. 3. 1. ... 1. 1. 1.]
label_train [0. 0. 1. ... 1. 1. 1.]
loaded 30207 events
data 11
label [[1. 0.]
 [1. 0.]
 [0. 1.]
 ...
 [0. 1.]
 [0. 1.]
 [0. 1.]]
len of X_train 11
len of X_val 11
Type of X_train <class 'list'>
Type of X_val <class 'list'>
Shape of X_train (11, 241658, 19, 1)
Shape of X_val (11, 30207, 19, 1)
Shape of Y_train (241658, 2)
Shape of Y_val (30207, 2)


# Data Processing

In [3]:
# Converting data lists to numpy arrays and reshaping
X_train_arr = np.array(X_train)
X_val_arr = np.array(X_val)
X_test_arr = np.array(X_test)

Y_train_arr = np.array(Y_train)
Y_val_arr = np.array(Y_val)
Y_test_arr = np.array(Y_test)

X_train_arr_reshaped = X_train_arr.reshape(len(X_train_arr[1]), 11, 19, 1)
X_val_arr_reshaped = X_val_arr.reshape(len(X_val_arr[1]), 11, 19, 1)
X_test_arr_reshaped = X_test_arr.reshape(len(X_test_arr[1]), 11, 19, 1)

print(X_train_arr_reshaped.shape)
print(Y_train_arr.shape)

(241658, 11, 19, 1)
(241658, 2)


In [4]:
# Convert the data into Alternative PyTorch tensors
X_alt_train = torch.tensor(X_train, dtype=torch.float32)
Y_alt_train = torch.tensor(Y_train, dtype=torch.long)
X_alt_val = torch.tensor(X_val, dtype=torch.float32)
Y_alt_val = torch.tensor(Y_val, dtype=torch.long)
X_alt_test = torch.tensor(X_test, dtype=torch.float32)
Y_alt_test = torch.tensor(Y_test, dtype=torch.long)

C:\Users\yoris\AppData\Local\Temp\ipykernel_17520\2023595757.py:2: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:248.)
  X_alt_train = torch.tensor(X_train, dtype=torch.float32)


In [5]:
print(X_alt_train.shape)
print(Y_alt_train.shape)

torch.Size([11, 241658, 19, 1])
torch.Size([241658, 2])


In [3]:
"""

# Converting our data to numpy arrays
X_train, X_val, X_test = np.array(X_train), np.array(X_val), np.array(X_test)
Y_train, Y_val, Y_test = np.array(Y_train), np.array(Y_val), np.array(Y_test),

# Compute mean and std of each feature
mean = np.mean(X_train)
std = np.std(X_train)

# Scale and normalize the data 
X_train_scaled = (X_train - mean) / std
X_val_scaled = (X_val - mean) / std
X_test_scaled = (X_test - mean) / std

"""

'\n\n# Converting our data to numpy arrays\nX_train, X_val, X_test = np.array(X_train), np.array(X_val), np.array(X_test)\nY_train, Y_val, Y_test = np.array(Y_train), np.array(Y_val), np.array(Y_test),\n\n# Compute mean and std of each feature\nmean = np.mean(X_train)\nstd = np.std(X_train)\n\n# Scale and normalize the data \nX_train_scaled = (X_train - mean) / std\nX_val_scaled = (X_val - mean) / std\nX_test_scaled = (X_test - mean) / std\n\n'

In [6]:
print("X_train_tensor:", list(X_train_tensor.size()))
print("Y_train_tensor:", list(y_train_tensor.size()))

print("X_val_tensor:", list(X_val_tensor.size()))
print("X_test_tensor:", list(X_test_tensor.size()))

NameError: name 'X_train_tensor' is not defined

## NEW ATTEMPT

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split


# Define the CNN model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=11, out_channels=32, kernel_size=3, stride=2, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(64 * 10 * 5, 128) # 1: Made *10 instead of *5
        self.fc2 = nn.Linear(128, 2)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 64 * 5 * 5) # 1: Made *10 instead of *5
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Create an instance of the model
model = CNN()


# Convert the data into PyTorch tensors
X_train_tensor = torch.tensor(X_train_arr_reshaped, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train_arr, dtype=torch.long)
X_val_tensor = torch.tensor(X_val_arr_reshaped, dtype=torch.float32)
Y_val_tensor = torch.tensor(Y_val_arr, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_arr_reshaped, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test_arr, dtype=torch.long)

"""
# Create an Alternative TensorDataset object for each set
train__alt_dataset = TensorDataset(X_alt_train, Y_alt_train)
val__alt_dataset = TensorDataset(X_alt_val, Y_alt_val)
test__alt_dataset = TensorDataset(X_alt_test, Y_alt_test)
"""

# Create a TensorDataset object for each set
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, Y_val_tensor)
test_dataset = TensorDataset(X_train_tensor, Y_train_tensor)

# Define the batch size
batch_size = 32

# Create a DataLoader object for each set
trainloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valloader = DataLoader(val_dataset, batch_size=batch_size)
testloader = DataLoader(test_dataset, batch_size=batch_size)

# Create an instance of the CNN model
model = CNN()

# Define the loss function and optimizer
#criterion = nn.CrossEntropyLoss()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


"""
# Train the model
for epoch in range(num_epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 2000 == 1999:    
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 2000))
            running_loss = 0.0
"""


# Train the model
num_epochs = 100
best_val_loss = float('inf')
early_stop_patience = 5
early_stop_counter = 0

for epoch in range(num_epochs):
    # Train on the training set
    running_train_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    # Evaluate on the validation set
    running_val_loss = 0.0
    with torch.no_grad():
        for data in valloader:
            inputs, labels = data
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

    # Compute average losses
    avg_train_loss = running_train_loss / len(trainloader)
    avg_val_loss = running_val_loss / len(valloader)

    # Print the average losses
    print('Epoch [%d/%d], train loss: %.3f, val loss: %.3f' %
          (epoch+1, num_epochs, avg_train_loss, avg_val_loss))

    # Check if the validation loss has improved
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stop_counter = 0
        # Save the best model
        torch.save(model.state_dict(), 'best_model_CNN.pt')
    else:
        early_stop_counter += 1

    # Check if early stopping is needed
    if early_stop_counter == early_stop_patience:
        print('Early stopping after epoch %d' % epoch)
        break

RuntimeError: Given input size: (32x10x1). Calculated output size: (32x5x0). Output size is too small

In [7]:
print(list(X_test_tensor.size()))
print(list(Y_test_tensor.size()))

[30207, 11, 19, 1]
[30207, 2]
